# Step 11: Spatial Microenvironment Analysis - Tumor Beds and Epithelia Beds

This notebook computes spatial relationships between cells and tissue regions to characterize tumor and normal epithelia microenvironments.

## Overview

The spatial analysis assigns each cell to a microenvironment "bed" (tumor or normal epithelia) and calculates signed distances to regional boundaries. This enables quantitative study of:

- **Tumor microenvironment**: Stromal composition near tumor regions
- **Normal epithelia microenvironment**: Stromal composition near epithelia regions
- **Cellular gradients**: How cell populations change with distance from tissue boundaries
- **Bed-specific clones**: Which specific tumor/epithelia regions cells associate with

## Input
- Unified CSV from Step 10 (phenotype_subsets column)
- Required spatial columns: `batch_id`, `tissue_roi`, `x_centroid`, `y_centroid`
- Required cell type columns: `tumor`, `epithelia`, `stroma`
- Required ROI columns: `tumor_roi`, `epithelia_roi`

## Output
- Updated unified CSV with 4 new columns:
  - `tumor_bed`: Binary assignment (1=tumor region, 0=normal epithelia region)
  - `distance_tumor`: Signed distance to tumor boundary
  - `distance_epithelia`: Signed distance to epithelia boundary
  - `bed_roi`: ROI label (t_N for tumor clones, e_N for epithelia clones)
- Summary statistics and distributions

## Key Features
- **Group-based processing**: Each (batch_id, tissue_roi) analyzed independently
- **Proximity assignment**: Stroma cells assigned to nearest tumor or epithelia
- **Signed distances**: Positive for stromal cells approaching regions, negative for cells within regions
- **Clone tracking**: Identify which specific tumor/epithelia clone each cell surrounds

## Imports and Setup

In [ ]:
# Imports & Environment Setup
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from spacemap.utils import log_time
from spacemap.spatial_beds_utils import run_spatial_beds_analysis

print(f"Project root: {project_root}")
print("Step 11: Spatial Beds Analysis Setup Complete")

## Configuration

Configure input/output paths and analysis parameters. Adjust these settings based on your data structure and analysis goals.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# I/O Paths
INPUT_DIR = project_root / 'outputs' / 'phenotyping'
OUTPUT_DIR = project_root / 'outputs' / 'phenotyping'

# Input file (unified phenotyping output from Step 10)
UNIFIED_INPUT_FILE = INPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'
UNIFIED_OUTPUT_FILE = OUTPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'

# Required columns for spatial analysis
REQUIRED_COLS = [
    'batch_id',           # Slide identifier
    'tissue_roi',         # Region of interest identifier
    'x_centroid',         # Cell x coordinate
    'y_centroid',         # Cell y coordinate
    'tumor',              # Binary: tumor cell (1) or not (0)
    'epithelia',          # Binary: epithelia cell (1) or not (0)
    'stroma',             # Binary: stromal cell (1) or not (0)
    'tumor_roi',          # Tumor region ID (0 if not in tumor)
    'epithelia_roi'       # Epithelia region ID (0 if not in epithelia)
]

# Processing options
VERBOSE = True            # Print detailed progress messages
NA_FILL_VALUE = 999       # Value for unassigned tumor_bed (tissue_roi with no tumor/epithelia)

print("Configuration loaded:")
print(f" Input file: {UNIFIED_INPUT_FILE}")
print(f" Output file: {UNIFIED_OUTPUT_FILE}")
print(f" Required columns: {len(REQUIRED_COLS)}")
print(f" Verbose output: {VERBOSE}")
print(f" NA fill value (unassigned): {NA_FILL_VALUE}")

## Step 1: Data Integrity Checks

Verify that the input file exists and contains all required columns for spatial analysis.

In [ ]:
# =============================================================================
# STEP 1: DATA INTEGRITY CHECKS
# =============================================================================
print("Step 1: Verifying data integrity...\n")

# Check file existence
if not UNIFIED_INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {UNIFIED_INPUT_FILE}")

# Load preview
preview_df = pd.read_csv(UNIFIED_INPUT_FILE, nrows=5000, low_memory=False)
print(f"Preview data: {len(preview_df):,} cells")

# Check required columns
existing_cols = [col for col in REQUIRED_COLS if col in preview_df.columns]
missing_cols = [col for col in REQUIRED_COLS if col not in preview_df.columns]

print(f"\nColumn verification:")
print(f"  Required: {len(REQUIRED_COLS)}")
print(f"  Found: {len(existing_cols)}")
if missing_cols:
    print(f"  MISSING: {missing_cols}")
    raise ValueError(f"Missing required columns: {missing_cols}")
else:
    print(f"  All required columns present ✓")

# Check cell type distribution
print(f"\nCell type distribution (preview):")
n_tumor = (preview_df['tumor'] == 1).sum()
n_epithelia = (preview_df['epithelia'] == 1).sum()
n_stroma = (preview_df['stroma'] == 1).sum()
n_other = len(preview_df) - n_tumor - n_epithelia - n_stroma

print(f"  Tumor cells: {n_tumor:,} ({n_tumor/len(preview_df)*100:.2f}%)")
print(f"  Epithelia cells: {n_epithelia:,} ({n_epithelia/len(preview_df)*100:.2f}%)")
print(f"  Stroma cells: {n_stroma:,} ({n_stroma/len(preview_df)*100:.2f}%)")
print(f"  Other cells: {n_other:,} ({n_other/len(preview_df)*100:.2f}%)")

# Check ROI distribution
print(f"\nROI distribution (preview):")
n_tumor_rois = (preview_df['tumor_roi'] > 0).sum()
n_epi_rois = (preview_df['epithelia_roi'] > 0).sum()
print(f"  Cells in tumor ROIs: {n_tumor_rois:,}")
print(f"  Cells in epithelia ROIs: {n_epi_rois:,}")

print("\n✓ Data integrity check passed")

## Step 2: Run Spatial Beds Analysis

Compute tumor_bed assignments, distance metrics, and bed_roi labels for all cells. This is the main analysis step.

In [ ]:
# =============================================================================
# STEP 2: RUN SPATIAL BEDS ANALYSIS
# =============================================================================
print("Step 2: Computing spatial microenvironment metrics...\n")

start_time = time.time()

# Run main analysis
result = run_spatial_beds_analysis(
    input_file=UNIFIED_INPUT_FILE,
    output_file=UNIFIED_OUTPUT_FILE,
    required_cols=REQUIRED_COLS,
    verbose=VERBOSE,
    na_fill_value=NA_FILL_VALUE
)

df = result['data']
stats = result['statistics']

elapsed_time = time.time() - start_time
print(f"\nCompleted in {elapsed_time/60:.1f} minutes")

## Step 3: Tumor Bed Distribution

Summarize the distribution of cells across tumor and normal epithelia microenvironments.

In [ ]:
# =============================================================================
# STEP 3: TUMOR BED DISTRIBUTION ANALYSIS
# =============================================================================
print("Step 3: Analyzing tumor_bed distribution...\n")

n_tumor_bed = (df['tumor_bed'] == 1.0).sum()
n_normal_bed = (df['tumor_bed'] == 0.0).sum()
n_unassigned = (df['tumor_bed'] == float(NA_FILL_VALUE)).sum()

print(f"{'='*80}")
print(f"TUMOR_BED DISTRIBUTION - OVERALL")
print(f"{'='*80}\n")

print(f"Tumor microenvironment (tumor_bed = 1):")
print(f"  Cells: {n_tumor_bed:,} ({n_tumor_bed/len(df)*100:.2f}%)")

print(f"\nNormal epithelia microenvironment (tumor_bed = 0):")
print(f"  Cells: {n_normal_bed:,} ({n_normal_bed/len(df)*100:.2f}%)")

print(f"\nUnassigned/No reference region (tumor_bed = {NA_FILL_VALUE}):")
print(f"  Cells: {n_unassigned:,} ({n_unassigned/len(df)*100:.2f}%)")
print(f"  (regions with no tumor or epithelia cells)")

# Distribution by batch_id with percentages
print(f"\n{'='*80}")
print(f"TUMOR_BED DISTRIBUTION BY BATCH_ID")
print(f"{'='*80}\n")

batch_stats = []
for batch_id in sorted(df['batch_id'].unique()):
    batch_df = df[df['batch_id'] == batch_id]
    n_batch = len(batch_df)
    n_tb = (batch_df['tumor_bed'] == 1.0).sum()
    n_nb = (batch_df['tumor_bed'] == 0.0).sum()
    n_ua = (batch_df['tumor_bed'] == float(NA_FILL_VALUE)).sum()
    
    batch_stats.append({
        'batch_id': batch_id,
        'Total': n_batch,
        'tumor_bed=1': n_tb,
        'pct_1': f"{n_tb/n_batch*100:.1f}%" if n_batch > 0 else "0.0%",
        'tumor_bed=0': n_nb,
        'pct_0': f"{n_nb/n_batch*100:.1f}%" if n_batch > 0 else "0.0%",
        'tumor_bed=999': n_ua,
        'pct_999': f"{n_ua/n_batch*100:.1f}%" if n_batch > 0 else "0.0%"
    })

batch_summary = pd.DataFrame(batch_stats)
print(batch_summary.to_string(index=False))

print(f"\n{'='*80}")

## Step 4: Distance to Tumor Analysis

Analyze signed distances to tumor regions for cells within the tumor microenvironment.

In [ ]:
# =============================================================================
# STEP 4: DISTANCE_TUMOR ANALYSIS
# =============================================================================
print("Step 4: Analyzing distance_tumor distribution...\n")

n_distance_tumor = df['distance_tumor'].notna().sum()
n_positive_tumor = (df['distance_tumor'] > 0).sum()
n_negative_tumor = (df['distance_tumor'] < 0).sum()

print(f"{'='*80}")
print(f"DISTANCE_TUMOR ANALYSIS")
print(f"{'='*80}\n")

print(f"Cells with distance_tumor values: {n_distance_tumor:,} ({n_distance_tumor/len(df)*100:.2f}%)")

if n_distance_tumor > 0:
    print(f"\nPositive distances (stroma → tumor boundary):")
    print(f"  Cells: {n_positive_tumor:,} ({n_positive_tumor/n_distance_tumor*100:.2f}%)")
    positive_distances = df[df['distance_tumor'] > 0]['distance_tumor']
    print(f"  Mean: {positive_distances.mean():.2f}")
    print(f"  Median: {positive_distances.median():.2f}")
    print(f"  Range: [{positive_distances.min():.2f}, {positive_distances.max():.2f}]")
    print(f"  Std: {positive_distances.std():.2f}")
    
    print(f"\nNegative distances (within tumor bed):")
    print(f"  Cells: {n_negative_tumor:,} ({n_negative_tumor/n_distance_tumor*100:.2f}%)")
    negative_distances = df[df['distance_tumor'] < 0]['distance_tumor']
    print(f"  Mean: {negative_distances.mean():.2f}")
    print(f"  Median: {negative_distances.median():.2f}")
    print(f"  Range: [{negative_distances.min():.2f}, {negative_distances.max():.2f}]")
    print(f"  Std: {negative_distances.std():.2f}")
    
    # Histogram
    fig, ax = plt.subplots(figsize=(10, 5))
    valid_distances = df['distance_tumor'].dropna()
    ax.hist(valid_distances, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Boundary (0)')
    ax.set_xlabel('Signed Distance to Tumor (pixels)', fontsize=12)
    ax.set_ylabel('Number of Cells', fontsize=12)
    ax.set_title('Distribution of Signed Distances to Tumor Regions', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print(f"\n{'='*80}")

## Step 5: Distance to Epithelia Analysis

Analyze signed distances to epithelia regions for cells within the normal epithelia microenvironment.

In [ ]:
# =============================================================================
# STEP 5: DISTANCE_EPITHELIA ANALYSIS
# =============================================================================
print("Step 5: Analyzing distance_epithelia distribution...\n")

n_distance_epi = df['distance_epithelia'].notna().sum()
n_positive_epi = (df['distance_epithelia'] > 0).sum()
n_negative_epi = (df['distance_epithelia'] < 0).sum()

print(f"{'='*80}")
print(f"DISTANCE_EPITHELIA ANALYSIS")
print(f"{'='*80}\n")

print(f"Cells with distance_epithelia values: {n_distance_epi:,} ({n_distance_epi/len(df)*100:.2f}%)")

if n_distance_epi > 0:
    print(f"\nPositive distances (stroma → epithelia boundary):")
    print(f"  Cells: {n_positive_epi:,} ({n_positive_epi/n_distance_epi*100:.2f}%)")
    positive_distances = df[df['distance_epithelia'] > 0]['distance_epithelia']
    print(f"  Mean: {positive_distances.mean():.2f}")
    print(f"  Median: {positive_distances.median():.2f}")
    print(f"  Range: [{positive_distances.min():.2f}, {positive_distances.max():.2f}]")
    print(f"  Std: {positive_distances.std():.2f}")
    
    print(f"\nNegative distances (within epithelia bed):")
    print(f"  Cells: {n_negative_epi:,} ({n_negative_epi/n_distance_epi*100:.2f}%)")
    negative_distances = df[df['distance_epithelia'] < 0]['distance_epithelia']
    print(f"  Mean: {negative_distances.mean():.2f}")
    print(f"  Median: {negative_distances.median():.2f}")
    print(f"  Range: [{negative_distances.min():.2f}, {negative_distances.max():.2f}]")
    print(f"  Std: {negative_distances.std():.2f}")
    
    # Histogram
    fig, ax = plt.subplots(figsize=(10, 5))
    valid_distances = df['distance_epithelia'].dropna()
    ax.hist(valid_distances, bins=50, edgecolor='black', alpha=0.7, color='coral')
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Boundary (0)')
    ax.set_xlabel('Signed Distance to Epithelia (pixels)', fontsize=12)
    ax.set_ylabel('Number of Cells', fontsize=12)
    ax.set_title('Distribution of Signed Distances to Epithelia Regions', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print(f"\n{'='*80}")

## Step 6: Bed ROI Assignment Analysis

Summarize the assignment of cells to specific tumor and epithelia clones (ROIs).

In [ ]:
# =============================================================================
# STEP 6: BED_ROI ASSIGNMENT ANALYSIS
# =============================================================================
print("Step 6: Analyzing bed_roi assignments...\n")

n_labeled = df['bed_roi'].notna().sum()
n_tumor_rois = df[df['bed_roi'].notna()]['bed_roi'].str.startswith('t_').sum()
n_epi_rois = df[df['bed_roi'].notna()]['bed_roi'].str.startswith('e_').sum()
n_unique_rois = df['bed_roi'].nunique()

print(f"{'='*80}")
print(f"BED_ROI ASSIGNMENT ANALYSIS")
print(f"{'='*80}\n")

print(f"Total cells with bed_roi labels: {n_labeled:,} ({n_labeled/len(df)*100:.2f}%)")
print(f"\nROI distribution:")
print(f"  Tumor bed ROIs (t_*): {n_tumor_rois:,} cells")
print(f"  Epithelia bed ROIs (e_*): {n_epi_rois:,} cells")
print(f"  Unique ROI labels: {n_unique_rois}")

# Top ROIs
print(f"\nTop 15 bed_roi labels:")
bed_roi_counts = df['bed_roi'].value_counts().head(15)
for i, (roi_label, count) in enumerate(bed_roi_counts.items(), 1):
    print(f"  {i:2d}. {str(roi_label):15s}: {count:8,} cells")

# Separate tumor and epithelia ROIs
tumor_rois = df[df['bed_roi'].notna()][df['bed_roi'].str.startswith('t_', na=False)]['bed_roi'].unique()
epi_rois = df[df['bed_roi'].notna()][df['bed_roi'].str.startswith('e_', na=False)]['bed_roi'].unique()

print(f"\nTumor bed ROIs (n={len(tumor_rois)}):")
if len(tumor_rois) <= 20:
    print(f"  {sorted(tumor_rois)}")
else:
    print(f"  {sorted(tumor_rois)[:20]} ... ({len(tumor_rois)} total)")

print(f"\nEpithelia bed ROIs (n={len(epi_rois)}):")
if len(epi_rois) <= 20:
    print(f"  {sorted(epi_rois)}")
else:
    print(f"  {sorted(epi_rois)[:20]} ... ({len(epi_rois)} total)")

print(f"\n{'='*80}")

## Step 7: Comprehensive Summary

Generate a comprehensive summary of all spatial metrics.

In [ ]:
# =============================================================================
# STEP 7: COMPREHENSIVE SUMMARY
# =============================================================================
print(f"\n{'='*80}")
print(f"SPATIAL MICROENVIRONMENT ANALYSIS - FINAL SUMMARY")
print(f"{'='*80}\n")

summary_data = {
    'Metric': [],
    'Count': [],
    'Percentage': []
}

# tumor_bed summary
summary_data['Metric'].append('Total cells analyzed')
summary_data['Count'].append(len(df))
summary_data['Percentage'].append(100.0)

summary_data['Metric'].append('  → Tumor microenvironment (tumor_bed=1)')
summary_data['Count'].append(n_tumor_bed)
summary_data['Percentage'].append(n_tumor_bed/len(df)*100 if len(df)>0 else 0)

summary_data['Metric'].append('  → Normal epithelia microenvironment (tumor_bed=0)')
summary_data['Count'].append(n_normal_bed)
summary_data['Percentage'].append(n_normal_bed/len(df)*100 if len(df)>0 else 0)

if n_blank > 0:
    summary_data['Metric'].append('  → Unclassified (special case)')
    summary_data['Count'].append(n_blank)
    summary_data['Percentage'].append(n_blank/len(df)*100)

summary_data['Metric'].append('\nDistance metrics computed')
summary_data['Count'].append('-')
summary_data['Percentage'].append('-')

summary_data['Metric'].append('  → distance_tumor (within tumor bed)')
summary_data['Count'].append(n_distance_tumor)
summary_data['Percentage'].append(n_distance_tumor/len(df)*100 if len(df)>0 else 0)

summary_data['Metric'].append('    · Stromal cells (positive distance)')
summary_data['Count'].append(n_positive_tumor)
summary_data['Percentage'].append(n_positive_tumor/n_distance_tumor*100 if n_distance_tumor>0 else 0)

summary_data['Metric'].append('    · Tumor cells (negative distance)')
summary_data['Count'].append(n_negative_tumor)
summary_data['Percentage'].append(n_negative_tumor/n_distance_tumor*100 if n_distance_tumor>0 else 0)

summary_data['Metric'].append('  → distance_epithelia (within normal bed)')
summary_data['Count'].append(n_distance_epi)
summary_data['Percentage'].append(n_distance_epi/len(df)*100 if len(df)>0 else 0)

summary_data['Metric'].append('    · Stromal cells (positive distance)')
summary_data['Count'].append(n_positive_epi)
summary_data['Percentage'].append(n_positive_epi/n_distance_epi*100 if n_distance_epi>0 else 0)

summary_data['Metric'].append('    · Epithelia cells (negative distance)')
summary_data['Count'].append(n_negative_epi)
summary_data['Percentage'].append(n_negative_epi/n_distance_epi*100 if n_distance_epi>0 else 0)

summary_data['Metric'].append('\nBed ROI assignments')
summary_data['Count'].append('-')
summary_data['Percentage'].append('-')

summary_data['Metric'].append('  → Total cells with ROI labels')
summary_data['Count'].append(n_labeled)
summary_data['Percentage'].append(n_labeled/len(df)*100 if len(df)>0 else 0)

summary_data['Metric'].append('  → Unique ROI identifiers')
summary_data['Count'].append(n_unique_rois)
summary_data['Percentage'].append('-')

summary_data['Metric'].append('  → Tumor bed clones (t_*)')
summary_data['Count'].append(len(tumor_rois))
summary_data['Percentage'].append('-')

summary_data['Metric'].append('  → Epithelia bed clones (e_*)')
summary_data['Count'].append(len(epi_rois))
summary_data['Percentage'].append('-')

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print(f"\n{'='*80}")
print(f"Output file: {UNIFIED_OUTPUT_FILE}")
print(f"File size: {UNIFIED_OUTPUT_FILE.stat().st_size / (1024*1024):.2f} MB")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"{'='*80}")

## Step 8: Example Data Inspection

Display representative rows showing new spatial columns for both tumor and normal bed cells.

In [ ]:
# =============================================================================
# STEP 8: EXAMPLE DATA INSPECTION
# =============================================================================
print("Step 8: Displaying example data...\n")

display_cols = [
    'cell_id', 'batch_id', 'tissue_roi', 'x_centroid', 'y_centroid',
    'tumor', 'epithelia', 'stroma',
    'tumor_bed', 'distance_tumor', 'distance_epithelia', 'bed_roi'
]
display_cols = [col for col in display_cols if col in df.columns]

# Tumor bed examples
tumor_bed_examples = df[df['tumor_bed'] == 1].head(5)
if len(tumor_bed_examples) > 0:
    print(f"{'='*80}")
    print(f"EXAMPLES: Tumor Microenvironment (tumor_bed = 1)")
    print(f"{'='*80}\n")
    print(tumor_bed_examples[display_cols].to_string(index=False))
else:
    print("No cells in tumor_bed == 1")

# Normal bed examples
normal_bed_examples = df[df['tumor_bed'] == 0].head(5)
if len(normal_bed_examples) > 0:
    print(f"\n{'='*80}")
    print(f"EXAMPLES: Normal Epithelia Microenvironment (tumor_bed = 0)")
    print(f"{'='*80}\n")
    print(normal_bed_examples[display_cols].to_string(index=False))
else:
    print("\nNo cells in tumor_bed == 0")

print(f"\n{'='*80}")

## Summary

Spatial microenvironment analysis completed successfully.

### Generated Columns
- **tumor_bed**: Binary assignment of cells to tumor (1) or normal epithelia (0) microenvironment
- **distance_tumor**: Signed distance to tumor region boundary (positive for stromal approach, negative for tumor cells)
- **distance_epithelia**: Signed distance to epithelia region boundary (positive for stromal approach, negative for epithelia cells)
- **bed_roi**: Clone identifier (t_N for tumor clones, e_N for epithelia clones)

### Outputs
- Updated unified CSV with all spatial columns
- Summary statistics and distributions
- Visualizations of distance metrics

### Next Steps
1. Analyze spatial composition of tumor vs normal microenvironments
2. Study cellular phenotypes at different distances from boundaries
3. Compare microenvironment composition across samples
4. Investigate clone-specific immune infiltration patterns